# Using GeoEnthalpyDelta component for alluvial fan simulation

Builds an alluvial fan where the location of the sediment feeder changes over time.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from landlab import RasterModelGrid
from landlab.components import GeoEnthalpyDelta, LinearDiffuser
from landlab.plot import imshow_grid

## Theory
> See the `geoenthalpy_delta` tutorial, which builds a simple delta fan, for details about the theory.

In this tutorial, we use the delta model for a terrestrial application: an alluvial fan. An alluvial fan has no sea level, so we set the `sea_level__elevation` field below the lowest point of the basement. Every node is then a topset node.

## Grid setup

A small 100 x 50 grid is enough for this example.

We use the x coordinate of each node to build the basement: a steep planar slope (`m1 = -1`) over the first third of the grid that breaks into a gentle, slightly uphill plain (`m2 = 0.02`). We then add some random roughness and smooth it slightly with Landlab's `LinearDiffuser`.

`GeoEnthalpyDelta` has no basement field. The initial `topographic__elevation` is the basement (there is no sediment yet), and the component tracks the sediment thickness on top of it.

In [ ]:
nrows, ncols = 100, 50
dx = dy = 0.2
grid = RasterModelGrid((nrows, ncols), xy_spacing=dx)

x = grid.x_of_node
m1 = -1
m2 = 0.02
break_at = ncols * dx / 3
topo = grid.add_field(
    "topographic__elevation",
    np.where(x <= break_at, m1 * x, m1 * break_at + m2 * (x - break_at)),
    at="node",
)
np.random.seed(50)
topo += np.random.rand(topo.size) / 2

# smooth the roughness with a single step of linear diffusion
diffuser = LinearDiffuser(grid, linear_diffusivity=0.05)
diffuser.run_one_step(0.5 * dx**2 / 0.05)

# GeoEnthalpyDelta transports sediment at every node, so make them all core
grid.status_at_node[:] = grid.BC_NODE_IS_CORE
initial_topography = topo.copy()

In [ ]:
imshow_grid(grid, "topographic__elevation", cmap="viridis")

In [ ]:
plt.plot(grid.x_of_node.reshape(grid.shape)[5, :], topo.reshape(grid.shape)[5, :])
plt.xlabel("x")
plt.ylabel("elevation")

## Feeding sediment with `sediment__influx`

Sediment enters the model only through the `sediment__influx` field: the volume of sediment per unit time added to each node. A feeder on the west edge is simply influx on some nodes of the first column (grid-x index 0).

The helper below spreads a total input rate `sediment_flux` evenly over the west-edge rows selected by a boolean `mask` (one entry per grid row). By default we use a single feeder of width 1 at the grid's lateral center.

> You aren't limited to a single, centered band, or even to the west edge. Because the component re-reads `sediment__influx` at every call to `run_one_step`, you can rewrite the field between calls. For example, if you have a record of how a river's terminus or distributary mouths migrate laterally over time, you can recompute the influx at each step (or at whatever interval your data resolves).

We don't have a continuous migration record here, so instead we build two different static masks, `mod_feeder_mask` and `another_mod_feeder_mask`, and switch between them at fixed steps inside the run loop below.

A couple of things to watch for when writing your own influx:

> Values must be non-negative and finite.

> Only the total `influx.sum()` enters the mass balance, so keep it fixed if you want the same total supply when you move the feeder.

In [ ]:
sediment_flux = 0.5  # total input rate, Q_in
influx = grid.add_zeros("sediment__influx", at="node")


def set_feeder(mask):
    influx_at_node = influx.reshape(grid.shape)
    influx_at_node[:] = 0.0
    influx_at_node[mask, 0] = sediment_flux / np.count_nonzero(mask)


y_of_row = grid.y_of_node.reshape(grid.shape)[:, 0]

In [ ]:
feeder_y_center = 0.5 * (y_of_row.min() + y_of_row.max())
default_feeder_mask = np.abs(y_of_row - feeder_y_center) <= 0.5 * 1.0 + 1.0e-12
set_feeder(default_feeder_mask)

Two feeders: one that is moved south of the center, plus a second, wider one farther north.

In [ ]:
mod_feeder_mask = np.zeros(nrows, dtype=bool)
mod_feeder_mask[27:34] = True
mod_feeder_mask[67:76] = True

In [ ]:
plt.imshow(mod_feeder_mask.reshape((nrows, 1)), cmap="plasma", aspect="auto")

In [ ]:
another_mod_feeder_mask = np.zeros(nrows, dtype=bool)
another_mod_feeder_mask[38:45] = True
another_mod_feeder_mask[67:76] = True

In [ ]:
plt.imshow(another_mod_feeder_mask.reshape((nrows, 1)), cmap="plasma", aspect="auto")

## Calling GeoEnthalpyDelta component

In [ ]:
# set below the lowest basement elevation (~ -3.3), so every node is topset
sea_level = grid.add_field("sea_level__elevation", -4.0, at="grid")

esd = GeoEnthalpyDelta(
    grid,
    topset_threshold=(0.3, 0.2),
    foreset_threshold=(2.0, 2.0),
    topset_diffusivity=(1.0, 1.0),
    foreset_diffusivity=(1.0, 1.0),
)

dt = 0.1  # run_one_step subcycles internally to stay CFL-stable
nsteps = 200
print(f"dt = {dt}, running {nsteps} steps (t_end = {nsteps * dt:.3g}) ...")

You should see `done in #.# s` once the run is completed.

In [ ]:
t0 = time.time()
volume_history = []
time_history = []

In [ ]:
for step in range(nsteps):
    if step == 40:
        set_feeder(mod_feeder_mask)
    elif step == 140:
        set_feeder(another_mod_feeder_mask)
    esd.run_one_step(dt)
    volume_history.append(np.sum(esd.sediment_thickness) * dx * dy)
    time_history.append(esd.time_elapsed)
print(f"done in {time.time() - t0:.2f} s")

## Mass Balance
Check that the total stored volume matches the total sediment supplied. Every feeder configuration delivers the same total rate, `sediment_flux`.

In [ ]:
expected_volume = sediment_flux * esd.time_elapsed
final_volume = np.sum(esd.sediment_thickness) * dx * dy
np.isclose(final_volume, expected_volume, rtol=1e-6)

Check the basement, sediment thickness, and topography

1. Basement elevation, `topographic__elevation - esd.sediment_thickness`

In [ ]:
basement = (grid.at_node["topographic__elevation"] - esd.sediment_thickness).reshape(
    grid.shape
)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    basement,
    origin="lower",
    extent=(0, ncols * dx, 0, nrows * dy),
    cmap="viridis",
    aspect="equal",
)
ax.set_title("Basement elevation (heatmap)")
ax.set_xlabel("x")
ax.set_ylabel("y")
fig.colorbar(im, ax=ax, label="elevation")
plt.show()

2. `esd.sediment_thickness`

In [ ]:
Hthickness = esd.sediment_thickness.reshape(grid.shape)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    Hthickness,
    origin="lower",
    extent=(0, ncols * dx, 0, nrows * dy),
    cmap="viridis",
    aspect="equal",
)
ax.set_title("Sediment thickness")
ax.set_xlabel("x")
ax.set_ylabel("y")
fig.colorbar(im, ax=ax, label="thickness")
plt.show()

3. `grid.at_node["topographic__elevation"]`

In [ ]:
final_topography = grid.at_node["topographic__elevation"].reshape(grid.shape)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    final_topography,
    origin="lower",
    extent=(0, ncols * dx, 0, nrows * dy),
    cmap="terrain",
    aspect="equal",
)
ax.set_title("Final topography elevation")
ax.set_xlabel("x")
ax.set_ylabel("y")
fig.colorbar(im, ax=ax, label="elevation")
plt.show()

## Comprehensive Plots

In [ ]:
H = Hthickness
eta = final_topography
final_sea_level = float(esd.sea_level)
X = grid.x_of_node.reshape(grid.shape)
Y = grid.y_of_node.reshape(grid.shape)
j_mid = nrows // 2
i_mid = ncols // 3

fig, axes = plt.subplots(4, 1, figsize=(5, 13))

im = axes[0].pcolormesh(X, Y, H, shading="auto", cmap="viridis")
axes[0].set_title("Sediment thickness")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_aspect("equal")
fig.colorbar(im, ax=axes[0], label="H")

axes[1].plot(X[j_mid], eta[j_mid], label="topography (basement + H)")
axes[1].plot(X[j_mid], basement[j_mid], "k--", label="basement")
axes[1].axhline(final_sea_level, color="tab:blue", linestyle=":", label="sea level")
axes[1].set_title("Centerline cross-section (along x)")
axes[1].set_xlabel("x")
axes[1].set_ylabel("elevation")
axes[1].legend(fontsize=8)

axes[2].plot(Y[:, i_mid], eta[:, i_mid], label="topography (basement + H)")
axes[2].plot(Y[:, i_mid], basement[:, i_mid], "k--", label="basement")
axes[2].set_title(f"Lateral cross-section at x = {X[0, i_mid]:.1f}")
axes[2].set_xlabel("y")
axes[2].set_ylabel("elevation")
axes[2].legend(fontsize=8)

axes[3].plot(time_history, volume_history, label="modeled volume")
axes[3].plot(
    time_history,
    [sediment_flux * t for t in time_history],
    "k--",
    label="expected volume",
)
axes[3].set_title("Mass balance")
axes[3].set_xlabel("time")
axes[3].set_ylabel("total volume")
axes[3].legend(fontsize=8)

fig.tight_layout()